# GUS04A — GeoTERYT Database Prerequisites (v5.0)

This notebook applies all v5.0 prerequisite changes to the GeoTERYT database:

1. **Extend year range** from 1988–2025 to 1986–2025
2. **Resolve historical TERYTs** — recover missing data from affiliated historical codes
3. **Create M_hh_size_1990 and M_hh_size_2000** — household size merged subjects
4. **Create M_age_1990** — age-only merged subject for Prediction1990
5. **Create M_educ_1990, M_educ_2000, M_educ_sex_1990, M_educ_sex_2000** — education merged subjects
6. **Build cross tables** for all new M_ subjects
7. **Validate** all prerequisites

**Input:** `geoteryt_O.pkl` (v4.3, optimized)  
**Output:** Updated database with extended year range, resolved historical data, and new M_ subjects

In [1]:
# ── Cell 1: Imports and database loading ──
import sys, os, pickle, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

# Paths
DB_PATH = Path(os.path.expanduser(
    '~/Documents/Studium Volkswirschaftslehre/3. Semester/'
    'Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_O.pkl'
))
TOOLS_PATH = DB_PATH.parents[2] / 'local_repo' / 'LRDWI-Paper' / 'Code' / 'tools'
sys.path.insert(0, str(TOOLS_PATH))

from geoTERYT_db import (
    load_complete_database, GeoTERYTDatabase, TERYTRecord,
    DataSeries, CrossTable, YEAR_RANGE_FULL, DATETIME_INDEX_FULL,
    _YEAR_BASE, _N_YEARS_FULL,
    LEVEL_VOIVODESHIP, LEVEL_POWIAT, LEVEL_GMINA,
    RODZ_SUB_DIVISIONS, RODZ_SUB_DIVISIONS_AND_DISTRICTS
)

print(f'YEAR_RANGE_FULL: {YEAR_RANGE_FULL[0]}–{YEAR_RANGE_FULL[-1]} ({len(YEAR_RANGE_FULL)} years)')
print(f'_YEAR_BASE: {_YEAR_BASE}')
print(f'_N_YEARS_FULL: {_N_YEARS_FULL}')

# Load database
db = load_complete_database(DB_PATH, verbose=True)
records = db._records
print(f'\nDatabase: {db}')
print(f'Total records: {len(records)}')

YEAR_RANGE_FULL: 1986–2025 (40 years)
_YEAR_BASE: 1986
_N_YEARS_FULL: 40
Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_O.pkl...
  Database version: 4.3
  ✓ Restored old voivodships: 49 rows
  ✓ Restored geometry store: 13,287 unique geometries
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4612 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3661
  ✓ Records with old_woj: 4104
  ✓ Records with data: 4584
  ✓ Records with cross tables: 4584
  ✓ Records with population data: 4533
  ✓ Records with pop_class: 3411

Database: GeoTERYTDatabase(4612 records, 1999-2024)
Total records: 4612


## Step 1: Extend year range to 1986–2025

The database currently spans 1988–2025. We extend to 1986–2025 (40 years) to accommodate
the Prediction1990 section which starts in 1986.

In [2]:
# ── Cell 2: Check current state and extend year range ──
sample_record = records['0201011']  # Bolesławiec
sample_key = list(sample_record.data.keys())[0]
print(f'Sample record: {sample_record.name} ({sample_record.teryt_id})')
print(f'Sample DataSeries index range: {sample_record.data[sample_key].values.index[0].year}–{sample_record.data[sample_key].values.index[-1].year}')
print(f'Sample pop index range: {sample_record.pop.index[0].year}–{sample_record.pop.index[-1].year}')
print(f'Pop values (first 5): {sample_record.pop.head()}')

if 'P2137' in sample_record.cross_tables:
    ct = sample_record.cross_tables['P2137']
    print(f'P2137 cross table year range: {ct.year_range[0]}–{ct.year_range[-1]}')
    print(f'P2137 years with data: {ct.years_with_data}')

Sample record: Bolesławiec (0201011)
Sample DataSeries index range: 1986–2025
Sample pop index range: 1986–2025
Pop values (first 5): 1986-01-01        NaN
1987-01-01        NaN
1988-01-01    43503.0
1989-01-01        NaN
1990-01-01        NaN
dtype: float64
P2137 cross table year range: 1988–2025
P2137 years with data: [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [3]:
# ── Cell 3: Apply year range extension ──
db.extend_year_range(new_start=1986, new_end=2025, verbose=True)

# Verify
print(f'\nAfter extension:')
print(f'Sample DataSeries index range: {sample_record.data[sample_key].values.index[0].year}–{sample_record.data[sample_key].values.index[-1].year}')
print(f'Sample pop index range: {sample_record.pop.index[0].year}–{sample_record.pop.index[-1].year}')
print(f'Sample pop length: {len(sample_record.pop)}')

if 'P2137' in sample_record.cross_tables:
    ct = sample_record.cross_tables['P2137']
    print(f'P2137 cross table year range: {ct.year_range[0]}–{ct.year_range[-1]}')
    print(f'P2137 years with data: {ct.years_with_data}')
    # Verify 1986-1987 are NaN
    t86 = ct.tables.get(1986)
    t87 = ct.tables.get(1987)
    print(f'1986 table: all NaN = {np.all(np.isnan(t86)) if t86 is not None else "missing"}')
    print(f'1987 table: all NaN = {np.all(np.isnan(t87)) if t87 is not None else "missing"}')

Extending year range from 1986-2025 to 1986-2025...
  ✓ Extended 4612 records to 1986-2025

After extension:
Sample DataSeries index range: 1986–2025
Sample pop index range: 1986–2025
Sample pop length: 40
P2137 cross table year range: 1986–2025
P2137 years with data: [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
1986 table: all NaN = True
1987 table: all NaN = True


## Step 2: Resolve historical TERYTs

Many gminas have missing data for years when their TERYT code was different.
The `resolve_historical_teryts()` method recovers this data by looking up
affiliated historical TERYT codes.

In [4]:
# ── Cell 4: Pre-resolution diagnostics ──
gminas = {tid: r for tid, r in records.items() if r.level == LEVEL_GMINA}
print(f'Total gminas: {len(gminas)}')

# Check how many gminas have historical codes
n_with_hist = sum(1 for r in gminas.values() if r.historical_codes)
n_multi_hist = sum(1 for r in gminas.values() if len(r.historical_codes) > 1)
print(f'Gminas with historical codes: {n_with_hist}')
print(f'Gminas with >1 historical code: {n_multi_hist}')

# Pre-resolution data coverage for key subjects
key_subjects = ['P2137', 'P2884', 'P2883', 'P2885', 'P2114', 'P2402', 'P2871', 'P2887']
print(f'\nPre-resolution data coverage (gminas):')
for sid in key_subjects:
    n_with = 0
    years_coverage = defaultdict(int)
    for r in gminas.values():
        subj_data = r.get_data_by_subject(sid)
        if subj_data:
            n_with += 1
            for key, ds in subj_data.items():
                for yr in ds.years:
                    years_coverage[yr] += 1
    if n_with > 0:
        years_str = ', '.join(f'{y}:{c}' for y, c in sorted(years_coverage.items())[:6])
        print(f'  {sid}: {n_with:5d} gminas | sample years: {years_str}...')

Total gminas: 4162
Gminas with historical codes: 4162
Gminas with >1 historical code: 440

Pre-resolution data coverage (gminas):
  P2137:  4134 gminas | sample years: 1995:172416, 1996:172800, 1997:173424, 1998:173952, 1999:173952, 2000:174432...
  P2884:  3624 gminas | sample years: 1988:28992...
  P2883:  3624 gminas | sample years: 1988:10872...
  P2885:  3624 gminas | sample years: 1988:14496...
  P2114:  3647 gminas | sample years: 2002:207879...
  P2402:  3647 gminas | sample years: 2002:87528...
  P2871:  3647 gminas | sample years: 2002:21882...
  P2887:  3624 gminas | sample years: 1988:14496...


In [5]:
# ── Cell 5: Run historical TERYT resolution ──
recovery = db.resolve_historical_teryts(verbose=True)

Resolving historical TERYTs for 19 subjects...
  P2137: recovered 336600 data points
  ✓ Total recovered: 336600 data points across 1 subjects


In [6]:
# ── Cell 6: Post-resolution diagnostics ──
print('Post-resolution data coverage (gminas):')
for sid in key_subjects:
    n_with = 0
    years_coverage = defaultdict(int)
    for r in gminas.values():
        subj_data = r.get_data_by_subject(sid)
        if subj_data:
            n_with += 1
            for key, ds in subj_data.items():
                for yr in ds.years:
                    years_coverage[yr] += 1
    if n_with > 0:
        years_str = ', '.join(f'{y}:{c}' for y, c in sorted(years_coverage.items())[:6])
        print(f'  {sid}: {n_with:5d} gminas | sample years: {years_str}...')

# Summary of recovery
print(f'\nRecovery summary:')
for sid in sorted(recovery.keys()):
    if recovery[sid] > 0:
        print(f'  {sid}: {recovery[sid]} data points recovered')

Post-resolution data coverage (gminas):
  P2137:  4134 gminas | sample years: 1995:182160, 1996:182544, 1997:183168, 1998:183696, 1999:183696, 2000:184176...
  P2884:  3624 gminas | sample years: 1988:28992...
  P2883:  3624 gminas | sample years: 1988:10872...
  P2885:  3624 gminas | sample years: 1988:14496...
  P2114:  3647 gminas | sample years: 2002:207879...
  P2402:  3647 gminas | sample years: 2002:87528...
  P2871:  3647 gminas | sample years: 2002:21882...
  P2887:  3624 gminas | sample years: 1988:14496...

Recovery summary:
  P2137: 336600 data points recovered


## Step 3: Create merged M_ subjects

Helper utilities for creating merged subjects with unified labels across censuses.

In [16]:
# ── Cell 7: Helper functions for creating M_ subjects ──

# Known dimensional structure per subject (dim_name -> semantic type)
# n1, n2 ordering depends on the source data column order
SUBJECT_DIMS = {
    'P2137': {'n1': 'age', 'n2': 'sex'},
    'P2114': {'n1': 'age', 'n2': 'sex'},
    'P2884': {'n1': 'age'},              # 1D age only
    'P2885': {'n1': 'educ'},             # 1D educ only
    'P2887': {'n1': 'hh_size'},          # 1D hh_size only
    'P2871': {'n2': 'hh_size'},          # 1D hh_size only — stored in n2!
    'P3420': {'n1': 'hh_size'},          # 1D hh_size only
    'P4287': {'n1': 'hh_size'},          # 1D hh_size only
    'P2402': {'n1': 'sex', 'n2': 'educ'},
    'P3309': {'n1': 'sex', 'n2': 'educ'},
    'P4315': {'n1': 'sex', 'n2': 'educ'},
    'P2350': {'n1': 'educ'},             # 1D educ only
    'P4092': {'n1': 'educ'},             # 1D educ only
    'H_age_sex': {'n1': 'age', 'n2': 'sex'},
    'H_sex_educ': {'n1': 'sex', 'n2': 'educ'},
}

def get_dim_for_type(subject_id, semantic_type):
    """Return the n-dim name for a given semantic type in a subject."""
    dims = SUBJECT_DIMS.get(subject_id, {})
    for dim_name, dim_type in dims.items():
        if dim_type == semantic_type:
            return dim_name
    return None

def extract_1d_labels(record, subject_id, dim_type, label_map, sum_groups=None):
    """Extract mapped 1D labels from a subject.
    
    Args:
        record: TERYTRecord
        subject_id: e.g. 'P2887'
        dim_type: semantic type of dimension to extract ('hh_size', 'age', 'educ')
        label_map: dict mapping source label (lowercase) -> unified label
        sum_groups: dict mapping unified_label -> [source_labels] for labels that need summing
    
    Returns:
        dict: unified_label -> pd.Series (values over time)
    """
    subj_data = record.get_data_by_subject(subject_id)
    if not subj_data:
        return {}
    
    dim_name = get_dim_for_type(subject_id, dim_type)
    if dim_name is None:
        return {}
    
    # Collect raw label -> values
    raw_labels = {}  # source_label -> pd.Series
    for key, ds in subj_data.items():
        if not ds.categories:
            continue
        label = ds.categories.get(dim_name)
        if label:
            raw_labels[label.strip()] = ds.values
    
    # Map to unified labels
    unified = {}
    for src_label, vals in raw_labels.items():
        mapped = label_map.get(src_label.lower())
        if mapped:
            if mapped not in unified:
                unified[mapped] = vals.copy()
            else:
                # Sum into existing (for multiple source labels → one unified)
                _add_series_inplace(unified[mapped], vals)
    
    # Handle sum groups (e.g., '3-4-osobowe' = '3 osoby' + '4 osoby')
    if sum_groups:
        for unified_label, source_labels in sum_groups.items():
            combined = pd.Series(data=np.nan, index=DATETIME_INDEX_FULL, dtype=float)
            for src_lbl in source_labels:
                for raw_lbl, vals in raw_labels.items():
                    if raw_lbl.lower() == src_lbl.lower():
                        _add_series_inplace(combined, vals)
            if combined.notna().any():
                unified[unified_label] = combined
    
    return unified

def extract_2d_filter_sex(record, subject_id, sex_value, educ_or_age_type, label_map, sum_groups=None):
    """Extract 1D labels from a 2D subject by filtering on sex dimension.
    
    Args:
        record: TERYTRecord
        subject_id: e.g. 'P2402'
        sex_value: sex label to filter by, e.g. 'ogółem'
        educ_or_age_type: 'educ' or 'age' — the other dimension type
        label_map: maps source label (lowercase) -> unified label for the non-sex dim
        sum_groups: dict mapping unified_label -> [source_labels] for labels needing summing
    
    Returns:
        dict: unified_label -> pd.Series
    """
    subj_data = record.get_data_by_subject(subject_id)
    if not subj_data:
        return {}
    
    sex_dim = get_dim_for_type(subject_id, 'sex')
    other_dim = get_dim_for_type(subject_id, educ_or_age_type)
    if sex_dim is None or other_dim is None:
        return {}
    
    # Collect raw other_label -> values, filtered by sex
    raw_labels = {}
    for key, ds in subj_data.items():
        if not ds.categories:
            continue
        sex_lbl = ds.categories.get(sex_dim, '')
        if sex_lbl.lower() != sex_value.lower():
            continue
        other_lbl = ds.categories.get(other_dim, '')
        if other_lbl:
            raw_labels[other_lbl.strip()] = ds.values
    
    # Map
    unified = {}
    for src_label, vals in raw_labels.items():
        mapped = label_map.get(src_label.lower())
        if mapped:
            if mapped not in unified:
                unified[mapped] = vals.copy()
            else:
                _add_series_inplace(unified[mapped], vals)
    
    if sum_groups:
        for unified_label, source_labels in sum_groups.items():
            combined = pd.Series(data=np.nan, index=DATETIME_INDEX_FULL, dtype=float)
            for src_lbl in source_labels:
                for raw_lbl, vals in raw_labels.items():
                    if raw_lbl.lower() == src_lbl.lower():
                        _add_series_inplace(combined, vals)
            if combined.notna().any():
                unified[unified_label] = combined
    
    return unified

def extract_2d_all_sex(record, subject_id, educ_or_age_type, label_map, sum_groups=None):
    """Extract 2D (other_label, sex_label) -> pd.Series from a 2D subject.
    
    Returns:
        dict: (unified_other_label, sex_label) -> pd.Series
    """
    subj_data = record.get_data_by_subject(subject_id)
    if not subj_data:
        return {}
    
    sex_dim = get_dim_for_type(subject_id, 'sex')
    other_dim = get_dim_for_type(subject_id, educ_or_age_type)
    if sex_dim is None or other_dim is None:
        return {}
    
    # Collect by (other_label, sex_label)
    raw_pairs = {}  # (other_label, sex_label) -> pd.Series
    for key, ds in subj_data.items():
        if not ds.categories:
            continue
        sex_lbl = ds.categories.get(sex_dim, '').strip().lower()
        other_lbl = ds.categories.get(other_dim, '').strip()
        if sex_lbl and other_lbl:
            raw_pairs[(other_lbl, sex_lbl)] = ds.values
    
    # Map other labels
    unified = {}
    for (other_lbl, sex_lbl), vals in raw_pairs.items():
        mapped = label_map.get(other_lbl.lower())
        if mapped:
            pk = (mapped, sex_lbl)
            if pk not in unified:
                unified[pk] = vals.copy()
            else:
                _add_series_inplace(unified[pk], vals)
    
    if sum_groups:
        # Get unique sex labels
        sex_labels_seen = set(sl for (_, sl) in raw_pairs.keys())
        for unified_label, source_labels in sum_groups.items():
            for sex_lbl in sex_labels_seen:
                combined = pd.Series(data=np.nan, index=DATETIME_INDEX_FULL, dtype=float)
                for src_lbl in source_labels:
                    for (raw_lbl, raw_sex), vals in raw_pairs.items():
                        if raw_lbl.lower() == src_lbl.lower() and raw_sex == sex_lbl:
                            _add_series_inplace(combined, vals)
                if combined.notna().any():
                    unified[(unified_label, sex_lbl)] = combined
    
    return unified

def _add_series_inplace(target, source):
    """Add source series values into target, treating NaN as 0 for initialized positions."""
    for ts in source.index:
        if not pd.isna(source[ts]):
            if pd.isna(target[ts]):
                target[ts] = source[ts]
            else:
                target[ts] += source[ts]

def store_1d_merged(record, subject_id, unified_labels, unified_data, subject_name=''):
    """Store unified 1D data as DataSeries on a record. Only fills NaN positions."""
    count = 0
    for i, label in enumerate(unified_labels):
        if label not in unified_data:
            continue
        var_id = f'M{i+1:04d}'
        mkey = ('Merged', subject_id, var_id)
        if mkey not in record.data:
            record.data[mkey] = DataSeries(
                source_type='Merged', subject_id=subject_id,
                variable_id=var_id, subject_name=subject_name,
                categories={'n1': label}
            )
        target = record.data[mkey]
        for ts, val in unified_data[label].dropna().items():
            if pd.isna(target.values.get(ts, np.nan)):
                target.values[ts] = val
        count += 1
    return count

def store_2d_merged(record, subject_id, dim1_labels, dim2_labels, pair_data, subject_name=''):
    """Store unified 2D data as DataSeries on a record. Only fills NaN positions.
    
    pair_data: dict of (dim1_label, dim2_label) -> pd.Series
    """
    count = 0
    var_idx = 0
    for d1_lbl in dim1_labels:
        for d2_lbl in dim2_labels:
            var_idx += 1
            pk = (d1_lbl, d2_lbl)
            if pk not in pair_data:
                continue
            vals = pair_data[pk]
            if not vals.notna().any():
                continue
            var_id = f'M{var_idx:04d}'
            mkey = ('Merged', subject_id, var_id)
            if mkey not in record.data:
                record.data[mkey] = DataSeries(
                    source_type='Merged', subject_id=subject_id,
                    variable_id=var_id, subject_name=subject_name,
                    categories={'n1': d1_lbl, 'n2': d2_lbl}
                )
            target = record.data[mkey]
            for ts, val in vals.dropna().items():
                if pd.isna(target.values.get(ts, np.nan)):
                    target.values[ts] = val
            count += 1
    return count

def compute_sum_label(unified, labels_to_sum, result_label):
    """Compute a sum label from existing unified labels."""
    combined = pd.Series(data=np.nan, index=DATETIME_INDEX_FULL, dtype=float)
    for lbl in labels_to_sum:
        if lbl in unified:
            _add_series_inplace(combined, unified[lbl])
    if combined.notna().any():
        unified[result_label] = combined

def compute_residual_label(unified, total_label, known_labels, result_label):
    """Compute residual = total - sum(known). Only for years where all parts are available."""
    total = unified.get(total_label)
    if total is None:
        return
    residual = pd.Series(data=np.nan, index=DATETIME_INDEX_FULL, dtype=float)
    for ts in total.index:
        t_val = total[ts]
        if pd.isna(t_val):
            continue
        parts_sum = 0.0
        for lbl in known_labels:
            v = unified.get(lbl, pd.Series(dtype=float)).get(ts, np.nan)
            if pd.isna(v):
                v = 0  # treat missing parts as 0 for residual computation
            parts_sum += v
        res = t_val - parts_sum
        if res >= 0:
            residual[ts] = res
    if residual.notna().any():
        unified[result_label] = residual

print('Helper functions defined.')
print(f'SUBJECT_DIMS P2871 = {SUBJECT_DIMS["P2871"]}')

Helper functions defined.
SUBJECT_DIMS P2871 = {'n2': 'hh_size'}


In [17]:
# ── Cell 8: Create M_hh_size_1990 ──
# P2887 (1988) + P2871 (2002), level=6 only
# Unified labels: ogółem, 1-osobowe, 2-osobowe, 3-4-osobowe, 5 i więcej-osobowe

M_HH_1990_LABELS = ['ogółem', '1-osobowe', '2-osobowe', '3-4-osobowe', '5 i więcej-osobowe']
SID_HH1990 = 'M_hh_size_1990'

# P2887 label map (1988): direct mapping, no ogółem in source
P2887_MAP = {
    '1-osobowe': '1-osobowe',
    '2-osobowe': '2-osobowe',
    '3-4-osobowe': '3-4-osobowe',
    '5 i więcej-osobowe': '5 i więcej-osobowe',
}

# P2871 label map (2002): direct mappings + sum group for 3-4-osobowe
P2871_1990_MAP = {
    '1 osoba': '1-osobowe',
    '2 osoby': '2-osobowe',
    '5 osób i więcej': '5 i więcej-osobowe',
    # ogółem mapped directly
    'ogółem': 'ogółem',
}
P2871_1990_SUM = {
    '3-4-osobowe': ['3 osoby', '4 osoby'],  # sum 3+4
}

n_created = 0
for tid, record in records.items():
    if record.level != LEVEL_GMINA:
        continue
    
    # P2887 (1988) — process first (priority)
    unified = extract_1d_labels(record, 'P2887', 'hh_size', P2887_MAP)
    if unified:
        # Compute ogółem = sum of all categories
        compute_sum_label(unified, 
                          ['1-osobowe', '2-osobowe', '3-4-osobowe', '5 i więcej-osobowe'], 
                          'ogółem')
        n_created += store_1d_merged(record, SID_HH1990, M_HH_1990_LABELS, unified, 'hh_size_1990')
    
    # P2871 (2002) — fills NaN positions only
    unified = extract_1d_labels(record, 'P2871', 'hh_size', P2871_1990_MAP, P2871_1990_SUM)
    if unified:
        n_created += store_1d_merged(record, SID_HH1990, M_HH_1990_LABELS, unified, 'hh_size_1990')

print(f'M_hh_size_1990: {n_created} DataSeries entries stored')

# Quick validation on sample
sample = records['0201011']
hh_data = sample.get_data_by_subject(SID_HH1990)
print(f'\nSample ({sample.name}): {len(hh_data)} variables')
for k, ds in sorted(hh_data.items()):
    yrs = ds.years
    vals = [f'{ds.values[pd.Timestamp(y,1,1)]:.0f}' for y in yrs]
    print(f'  {list(ds.categories.values())}: years={yrs}, values={vals}')

M_hh_size_1990: 36355 DataSeries entries stored

Sample (Bolesławiec): 5 variables
  ['ogółem']: years=[1988, 2002], values=['14291', '15793']
  ['1-osobowe']: years=[1988, 2002], values=['2360', '3913']
  ['2-osobowe']: years=[1988, 2002], values=['3467', '4246']
  ['3-4-osobowe']: years=[1988, 2002], values=['6850', '6384']
  ['5 i więcej-osobowe']: years=[1988, 2002], values=['1614', '1250']


In [ ]:
# ── Cell 8b: Subject dimension & label verification ──
# Discovery results (verified at runtime):
# P2871: dimension n2 (not n1!) for hh_size — fixed in SUBJECT_DIMS above
# P3420: level=5 only (powiat), n1=hh_size
# P3309: level=5 only (powiat), n1=sex, n2=educ
# P2350, P4092: level=2 only (voivodeship), n1=educ
# H_age_sex: level 0+2 only (country + old voivodeships)
# H_sex_educ: level 0 only (country)
print('Dimension/label verification notes — see comments above.')

H_age_sex: level distribution = {0: 1, 2: 49}
H_sex_educ: level distribution = {0: 1}


In [18]:
# ── Cell 9: Create M_hh_size_2000 ──
# P2871 (2002, gmina) + P3420 (2011, powiat) + P4287 (2021, gmina)
# Unified labels: ogółem, 1-osobowe, 2-osobowe, 3-osobowe, 4-osobowe, 5 i więcej-osobowe

M_HH_2000_LABELS = ['ogółem', '1-osobowe', '2-osobowe', '3-osobowe', '4-osobowe', '5 i więcej-osobowe']
SID_HH2000 = 'M_hh_size_2000'

P2871_2000_MAP = {
    'ogółem': 'ogółem',
    '1 osoba': '1-osobowe',
    '2 osoby': '2-osobowe',
    '3 osoby': '3-osobowe',
    '4 osoby': '4-osobowe',
    '5 osób i więcej': '5 i więcej-osobowe',
}

P3420_MAP = {
    'ogółem': 'ogółem',
    '1-osobowe': '1-osobowe',
    '2-osobowe': '2-osobowe',
    '3-osobowe': '3-osobowe',
    '4-osobowe': '4-osobowe',
    '5-osobowe i większe': '5 i więcej-osobowe',
}

P4287_MAP = {
    'ogółem': 'ogółem',
    'gospodarstwa domowe 1-osobowe': '1-osobowe',
    'gospodarstwa domowe 2-osobowe': '2-osobowe',
    'gospodarstwa domowe 3-osobowe': '3-osobowe',
    'gospodarstwa domowe 4-osobowe': '4-osobowe',
    'gospodarstwa domowe 5-osobowe i większe': '5 i więcej-osobowe',
}

n_created = 0
for tid, record in records.items():
    # P2871 (2002, level=6 gmina)
    if record.level == LEVEL_GMINA:
        unified = extract_1d_labels(record, 'P2871', 'hh_size', P2871_2000_MAP)
        if unified:
            n_created += store_1d_merged(record, SID_HH2000, M_HH_2000_LABELS, unified, 'hh_size_2000')
    
    # P3420 (2011, level=5 powiat)
    if record.level == LEVEL_POWIAT:
        unified = extract_1d_labels(record, 'P3420', 'hh_size', P3420_MAP)
        if unified:
            n_created += store_1d_merged(record, SID_HH2000, M_HH_2000_LABELS, unified, 'hh_size_2000')
    
    # P4287 (2021, level=6 gmina)
    if record.level == LEVEL_GMINA:
        unified = extract_1d_labels(record, 'P4287', 'hh_size', P4287_MAP)
        if unified:
            n_created += store_1d_merged(record, SID_HH2000, M_HH_2000_LABELS, unified, 'hh_size_2000')

print(f'M_hh_size_2000: {n_created} DataSeries entries stored')

# Quick validation
for sample_tid in ['0201011', '0201000']:  # gmina and powiat
    sample = records.get(sample_tid)
    if sample:
        hh_data = sample.get_data_by_subject(SID_HH2000)
        if hh_data:
            print(f'\nSample ({sample.name}, level={sample.level}): {len(hh_data)} variables')
            for k, ds in sorted(hh_data.items()):
                yrs = ds.years
                vals = [f'{ds.values[pd.Timestamp(y,1,1)]:.0f}' for y in yrs]
                print(f'  {list(ds.categories.values())}: years={yrs}, values={vals}')

M_hh_size_2000: 46944 DataSeries entries stored

Sample (Bolesławiec, level=6): 6 variables
  ['ogółem']: years=[2002, 2021], values=['15793', '14414']
  ['1-osobowe']: years=[2002, 2021], values=['3913', '3878']
  ['2-osobowe']: years=[2002, 2021], values=['4246', '4247']
  ['3-osobowe']: years=[2002, 2021], values=['3505', '2909']
  ['4-osobowe']: years=[2002, 2021], values=['2879', '2012']
  ['5 i więcej-osobowe']: years=[2002, 2021], values=['1250', '1368']

Sample (bolesławiecki, level=5): 6 variables
  ['ogółem']: years=[2011], values=['31463']
  ['1-osobowe']: years=[2011], values=['6773']
  ['2-osobowe']: years=[2011], values=['8458']
  ['3-osobowe']: years=[2011], values=['6487']
  ['4-osobowe']: years=[2011], values=['5294']
  ['5 i więcej-osobowe']: years=[2011], values=['4450']


In [19]:
# ── Cell 10: Create M_age_1990 ──
# P2884 (1988, gmina) + P2137 (sex=ogółem, gmina, aggregate 5yr→10yr)
# + H_age_sex (sex=ogółem, old voivodeships, aggregate to 10yr)
# Unified labels: ogółem, 0-9, 10-19, 20-29, 30-39, 40-49, 50-59, 60 lat i więcej

M_AGE_1990_LABELS = ['ogółem', '0-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60 lat i więcej']
SID_AGE1990 = 'M_age_1990'

# P2884 has 10-year bins already (direct mapping)
P2884_MAP = {
    'ogółem': 'ogółem',
    '0-9': '0-9',
    '10-19': '10-19',
    '20-29': '20-29',
    '30-39': '30-39',
    '40-49': '40-49',
    '50-59': '50-59',
    '60 lat i więcej': '60 lat i więcej',
}

# P2137 5-year bins → aggregate to 10-year bins
P2137_AGE_MAP = {
    'ogółem': 'ogółem',
    # Fine bins are NOT mapped directly — they're summed via sum_groups
}
P2137_AGE_SUM = {
    '0-9':   ['0-4', '5-9'],
    '10-19': ['10-14', '15-19'],
    '20-29': ['20-24', '25-29'],
    '30-39': ['30-34', '35-39'],
    '40-49': ['40-44', '45-49'],
    '50-59': ['50-54', '55-59'],
    '60 lat i więcej': ['60-64', '65-69', '70 i więcej'],
}

# H_age_sex 5-year bins (with separate 0 and 1-4) → 10-year bins
HAGE_MAP = {
    'ogółem': 'ogółem',
}
HAGE_SUM = {
    '0-9':   ['0', '1-4', '5-9'],
    '10-19': ['10-14', '15-19'],
    '20-29': ['20-24', '25-29'],
    '30-39': ['30-34', '35-39'],
    '40-49': ['40-44', '45-49'],
    '50-59': ['50-54', '55-59'],
    '60 lat i więcej': ['60-64', '65-69', '70 i więcej'],
}

n_created = 0
for tid, record in records.items():
    # P2884 (1988, level=6 gmina) — direct 10yr bins
    if record.level == LEVEL_GMINA:
        unified = extract_1d_labels(record, 'P2884', 'age', P2884_MAP)
        if unified:
            n_created += store_1d_merged(record, SID_AGE1990, M_AGE_1990_LABELS, unified, 'age_1990')
    
    # P2137 (1995-2024, level=6 gmina, sex=ogółem, aggregate 5yr→10yr)
    if record.level == LEVEL_GMINA:
        unified = extract_2d_filter_sex(record, 'P2137', 'ogółem', 'age',
                                         P2137_AGE_MAP, P2137_AGE_SUM)
        if unified:
            n_created += store_1d_merged(record, SID_AGE1990, M_AGE_1990_LABELS, unified, 'age_1990')
    
    # H_age_sex (1986-1994, level=2 old voivodeships, sex=ogółem, aggregate to 10yr)
    h_data = record.get_data_by_subject('H_age_sex')
    if h_data:
        unified = extract_2d_filter_sex(record, 'H_age_sex', 'ogółem', 'age',
                                         HAGE_MAP, HAGE_SUM)
        if unified:
            n_created += store_1d_merged(record, SID_AGE1990, M_AGE_1990_LABELS, unified, 'age_1990')

print(f'M_age_1990: {n_created} DataSeries entries stored')

# Validation - check different levels
for sample_tid in ['0201011', '6100000']:  # gmina + old voivodeship
    sample = records.get(sample_tid)
    if sample:
        age_data = sample.get_data_by_subject(SID_AGE1990)
        if age_data:
            print(f'\nSample ({sample.name}, level={sample.level}): {len(age_data)} variables')
            for k, ds in sorted(age_data.items()):
                yrs = ds.years[:5]
                vals = [f'{ds.values[pd.Timestamp(y,1,1)]:.0f}' for y in yrs]
                print(f'  {list(ds.categories.values())}: first years={yrs}, values={vals}')

M_age_1990: 62464 DataSeries entries stored

Sample (Bolesławiec, level=6): 8 variables
  ['ogółem']: first years=[1988, 1995, 1996, 1997, 1998], values=['43503', '44436', '44298', '44184', '44026']
  ['0-9']: first years=[1988, 1995, 1996, 1997, 1998], values=['7189', '5186', '4919', '4714', '4480']
  ['10-19']: first years=[1988, 1995, 1996, 1997, 1998], values=['6856', '7702', '7530', '7310', '7072']
  ['20-29']: first years=[1988, 1995, 1996, 1997, 1998], values=['5597', '5662', '5912', '6175', '6408']
  ['30-39']: first years=[1988, 1995, 1996, 1997, 1998], values=['8679', '6901', '6501', '6095', '5746']
  ['40-49']: first years=[1988, 1995, 1996, 1997, 1998], values=['5364', '8081', '8335', '8388', '8403']
  ['50-59']: first years=[1988, 1995, 1996, 1997, 1998], values=['5124', '4534', '4495', '4664', '4939']
  ['60 lat i więcej']: first years=[1988, 1995, 1996, 1997, 1998], values=['4669', '6370', '6606', '6838', '6978']

Sample (Tarnobrzeskie, level=2): 8 variables
  ['ogółem']

In [20]:
# ── Cell 11: Create M_educ_1990 ──
# P2885 (1988, gmina) + P2402 (2002, gmina, sex=ogółem) + H_sex_educ (country, sex=ogółem)
# Labels: ogółem, wyższe, średnie, zasadnicze zawodowe, podstawowe, 
#         podstawowe nieukończone i bez wykształcenia

M_EDUC_1990_LABELS = [
    'ogółem', 'wyższe', 'średnie', 'zasadnicze zawodowe',
    'podstawowe', 'podstawowe nieukończone i bez wykształcenia'
]
SID_EDUC1990 = 'M_educ_1990'

# P2885 (1988): direct 1D educ labels
P2885_MAP = {
    'wyższe': 'wyższe',
    'średnie': 'średnie',
    'zasadnicze zawodowe': 'zasadnicze zawodowe',
    'podstawowe': 'podstawowe',
}

# P2402 (2002, sex=ogółem): map to M_educ_1990 labels
P2402_1990_MAP = {
    'wyższe': 'wyższe',
    'policealne': 'średnie',          # policealne → średnie
    'średnie razem': 'średnie',       # średnie razem → średnie (summed with policealne)
    'zasadnicze zawodowe': 'zasadnicze zawodowe',
    'podstawowe ukończone': 'podstawowe',
    'podstawowe nieukończone i bez wykształcenia': 'podstawowe nieukończone i bez wykształcenia',
}

# H_sex_educ (country level, sex=ogółem)
H_EDUC_1990_MAP = {
    'ogółem': 'ogółem',
    'wyższe': 'wyższe',
    'średnie': 'średnie',
    'zasadnicze zawodowe': 'zasadnicze zawodowe',
    'podstawowe': 'podstawowe',
    'niepełne podstawowe i bez wykształcenia': 'podstawowe nieukończone i bez wykształcenia',
}

n_created = 0
for tid, record in records.items():
    # ── P2885 (1988, level=6) ──
    if record.level == LEVEL_GMINA:
        unified = extract_1d_labels(record, 'P2885', 'educ', P2885_MAP)
        if unified:
            # Compute ogółem for 1988 from P2884 (population 15+)
            # ogółem_15plus = [10-19]*0.5 + [20-29] + [30-39] + [40-49] + [50-59] + [60+]
            p2884_data = record.get_data_by_subject('P2884')
            if p2884_data:
                ts_1988 = pd.Timestamp(1988, 1, 1)
                age_vals = {}
                for key, ds in p2884_data.items():
                    if ds.categories:
                        lbl = list(ds.categories.values())[0].strip().lower()
                        v = ds.values.get(ts_1988, np.nan)
                        if not pd.isna(v):
                            age_vals[lbl] = v
                
                total_15plus = 0.0
                have_data = False
                for lbl, v in age_vals.items():
                    if '10-19' in lbl:
                        total_15plus += v * 0.5  # approximate 15-19 as half of 10-19
                        have_data = True
                    elif any(x in lbl for x in ['20-29', '30-39', '40-49', '50-59', '60']):
                        if lbl != 'ogółem':
                            total_15plus += v
                            have_data = True
                
                if have_data:
                    og_series = pd.Series(data=np.nan, index=DATETIME_INDEX_FULL, dtype=float)
                    og_series[ts_1988] = total_15plus
                    unified['ogółem'] = og_series
            
            # Compute 'podstawowe nieukończone' as residual for 1988
            if 'ogółem' in unified:
                compute_residual_label(unified, 'ogółem',
                    ['wyższe', 'średnie', 'zasadnicze zawodowe', 'podstawowe'],
                    'podstawowe nieukończone i bez wykształcenia')
            
            n_created += store_1d_merged(record, SID_EDUC1990, M_EDUC_1990_LABELS, unified, 'educ_1990')
    
    # ── P2402 (2002, level=6, sex=ogółem) ──
    if record.level == LEVEL_GMINA:
        unified = extract_2d_filter_sex(record, 'P2402', 'ogółem', 'educ', P2402_1990_MAP)
        if unified:
            # Compute ogółem for 2002 from P2114 (ages 15+)
            # P2114 is enclosed in P2137, has ogółem for age 15+
            p2114_data = record.get_data_by_subject('P2114')
            if p2114_data:
                ts_2002 = pd.Timestamp(2002, 1, 1)
                # P2114 has n1=age, n2=sex; filter sex=ogółem and sum ages 15+
                sex_dim = get_dim_for_type('P2114', 'sex')
                age_dim = get_dim_for_type('P2114', 'age')
                if sex_dim and age_dim:
                    total_15plus = 0.0
                    have_data = False
                    for key, ds in p2114_data.items():
                        if not ds.categories:
                            continue
                        sex_lbl = ds.categories.get(sex_dim, '')
                        age_lbl = ds.categories.get(age_dim, '').strip().lower()
                        if sex_lbl.lower() != 'ogółem':
                            continue
                        v = ds.values.get(ts_2002, np.nan)
                        if pd.isna(v):
                            continue
                        # Include ages 15+: 15-19, 20-24, ..., 80-84, 85 i więcej
                        if age_lbl == 'ogółem':
                            continue
                        try:
                            lower = int(age_lbl.split('-')[0].split(' ')[0])
                            if lower >= 15:
                                total_15plus += v
                                have_data = True
                        except ValueError:
                            if 'więcej' in age_lbl:
                                total_15plus += v
                                have_data = True
                    
                    if have_data:
                        og_series = pd.Series(data=np.nan, index=DATETIME_INDEX_FULL, dtype=float)
                        og_series[ts_2002] = total_15plus
                        if 'ogółem' not in unified:
                            unified['ogółem'] = og_series
                        else:
                            _add_series_inplace(unified['ogółem'], og_series)
            
            n_created += store_1d_merged(record, SID_EDUC1990, M_EDUC_1990_LABELS, unified, 'educ_1990')
    
    # ── H_sex_educ (country level, sex=ogółem) ──
    h_data = record.get_data_by_subject('H_sex_educ')
    if h_data:
        unified = extract_2d_filter_sex(record, 'H_sex_educ', 'ogółem', 'educ', H_EDUC_1990_MAP)
        if unified:
            # Compute 'niepełne' as residual for years without it
            if 'ogółem' in unified and 'podstawowe nieukończone i bez wykształcenia' not in unified:
                compute_residual_label(unified, 'ogółem',
                    ['wyższe', 'średnie', 'zasadnicze zawodowe', 'podstawowe'],
                    'podstawowe nieukończone i bez wykształcenia')
            n_created += store_1d_merged(record, SID_EDUC1990, M_EDUC_1990_LABELS, unified, 'educ_1990')

print(f'M_educ_1990: {n_created} DataSeries entries stored')

# Validation
for sample_tid in ['0201011', '0000000']:  # gmina + country
    sample = records.get(sample_tid)
    if sample:
        educ_data = sample.get_data_by_subject(SID_EDUC1990)
        if educ_data:
            print(f'\nSample ({sample.name}, level={sample.level}): {len(educ_data)} variables')
            for k, ds in sorted(educ_data.items()):
                yrs = ds.years
                vals = [f'{ds.values[pd.Timestamp(y,1,1)]:.0f}' for y in yrs]
                print(f'  {list(ds.categories.values())}: years={yrs}, values={vals}')

M_educ_1990: 39975 DataSeries entries stored

Sample (Bolesławiec, level=6): 6 variables
  ['ogółem']: years=[1988], values=['32861']
  ['wyższe']: years=[1988, 2002], values=['2213', '3543']
  ['średnie']: years=[1988, 2002], values=['9786', '13472']
  ['zasadnicze zawodowe']: years=[1988, 2002], values=['7924', '8595']
  ['podstawowe']: years=[1988, 2002], values=['11100', '9787']
  ['podstawowe nieukończone i bez wykształcenia']: years=[1988, 2002], values=['1838', '755']

Sample (Poland, level=0): 6 variables
  ['ogółem']: years=[1988, 1991, 1992, 1993, 1994], values=['28268775', '28899000', '29141000', '29393000', '29658000']
  ['wyższe']: years=[1988, 1991, 1992, 1993, 1994], values=['1838331', '1950000', '1999000', '2055000', '2111000']
  ['średnie']: years=[1988, 1991, 1992, 1993, 1994], values=['6979573', '7424000', '7594000', '7773000', '7976000']
  ['zasadnicze zawodowe']: years=[1988, 1991, 1992, 1993, 1994], values=['6665767', '7114000', '7269000', '7407000', '7536000']
  

In [21]:
# ── Cell 12: Create M_educ_2000 ──
# P2402 (2002, gmina) + P3309 (2011, powiat) + P4315 (2021, gmina) ← census priority
# P2350 (1995-2020, voivodeship) + P4092 (2010-2024, voivodeship) ← lower priority
# Sex=ogółem only (1D educ). Labels:
# wyższe, policealne oraz średnie zawodowe/branżowe, średnie ogólnokształcące,
# zasadnicze zawodowe/branżowe, gimnazjalne podstawowe i niższe

M_EDUC_2000_LABELS = [
    'wyższe',
    'policealne oraz średnie zawodowe/branżowe',
    'średnie ogólnokształcące',
    'zasadnicze zawodowe/branżowe',
    'gimnazjalne, podstawowe i niższe',
]
SID_EDUC2000 = 'M_educ_2000'

# P2402 (2002, sex=ogółem) → policealne + średnie zawodowe merged, basics merged
P2402_2000_MAP = {
    'wyższe': 'wyższe',
    'policealne': 'policealne oraz średnie zawodowe/branżowe',          # → sum
    'średnie zawodowe': 'policealne oraz średnie zawodowe/branżowe',    # → sum
    'średnie ogólnokształcące': 'średnie ogólnokształcące',
    'zasadnicze zawodowe': 'zasadnicze zawodowe/branżowe',
    'podstawowe ukończone': 'gimnazjalne, podstawowe i niższe',                        # → sum
    'podstawowe nieukończone i bez wykształcenia': 'gimnazjalne, podstawowe i niższe',  # → sum
}

# P3309 (2011, sex=ogółem)
P3309_2000_MAP = {
    'wyższe': 'wyższe',
    'średnie i policealne - średnie zawodowe': 'policealne oraz średnie zawodowe/branżowe',
    'średnie i policealne - średnie ogólnokształcące': 'średnie ogólnokształcące',
    'zasadnicze zawodowe': 'zasadnicze zawodowe/branżowe',
    'gimnazjalne': 'gimnazjalne, podstawowe i niższe',                                       # → sum
    'podstawowe ukończone': 'gimnazjalne, podstawowe i niższe',                              # → sum
    'podstawowe nieukończone i bez wykształcenia szkolnego': 'gimnazjalne, podstawowe i niższe',  # → sum
}

# P4315 (2021, sex=ogółem)
P4315_2000_MAP = {
    'wyższe': 'wyższe',
    'średnie i policealne - średnie zawodowe': 'policealne oraz średnie zawodowe/branżowe',
    'średnie i policealne - średnie ogólnokształcące': 'średnie ogólnokształcące',
    'zasadnicze zawodowe/branżowe': 'zasadnicze zawodowe/branżowe',
    'gimnazjalne': 'gimnazjalne, podstawowe i niższe',
    'podstawowe ukończone': 'gimnazjalne, podstawowe i niższe',
    'podstawowe nieukończone i bez wykształcenia szkolnego': 'gimnazjalne, podstawowe i niższe',
    'nieustalony': 'gimnazjalne, podstawowe i niższe',  # lump with lowest
}

# P2350, P4092 (voivodeship, direct labels)
P2350_MAP = {
    'wyższe': 'wyższe',
    'policealne oraz średnie zawodowe/branżowe': 'policealne oraz średnie zawodowe/branżowe',
    'średnie ogólnokształcące': 'średnie ogólnokształcące',
    'zasadnicze zawodowe/branżowe': 'zasadnicze zawodowe/branżowe',
    'gimnazjalne, podstawowe i niższe': 'gimnazjalne, podstawowe i niższe',
}

n_created = 0

# PASS 1: Census data (priority) — P2402, P3309, P4315
for tid, record in records.items():
    if record.level == LEVEL_GMINA:
        # P2402 (2002)
        unified = extract_2d_filter_sex(record, 'P2402', 'ogółem', 'educ', P2402_2000_MAP)
        if unified:
            n_created += store_1d_merged(record, SID_EDUC2000, M_EDUC_2000_LABELS, unified, 'educ_2000')
        
        # P4315 (2021)
        unified = extract_2d_filter_sex(record, 'P4315', 'ogółem', 'educ', P4315_2000_MAP)
        if unified:
            n_created += store_1d_merged(record, SID_EDUC2000, M_EDUC_2000_LABELS, unified, 'educ_2000')
    
    if record.level == LEVEL_POWIAT:
        # P3309 (2011)
        unified = extract_2d_filter_sex(record, 'P3309', 'ogółem', 'educ', P3309_2000_MAP)
        if unified:
            n_created += store_1d_merged(record, SID_EDUC2000, M_EDUC_2000_LABELS, unified, 'educ_2000')

# PASS 2: BDL data (lower priority, fills remaining NaN)
for tid, record in records.items():
    if record.level == LEVEL_VOIVODESHIP or tid == '0000000':
        # P2350
        unified = extract_1d_labels(record, 'P2350', 'educ', P2350_MAP)
        if unified:
            n_created += store_1d_merged(record, SID_EDUC2000, M_EDUC_2000_LABELS, unified, 'educ_2000')
        
        # P4092
        unified = extract_1d_labels(record, 'P4092', 'educ', P2350_MAP)  # same labels
        if unified:
            n_created += store_1d_merged(record, SID_EDUC2000, M_EDUC_2000_LABELS, unified, 'educ_2000')

print(f'M_educ_2000: {n_created} DataSeries entries stored')

# Validation
for sample_tid in ['0201011', '0201000', '0200000']:
    sample = records.get(sample_tid)
    if sample:
        educ_data = sample.get_data_by_subject(SID_EDUC2000)
        if educ_data:
            print(f'\nSample ({sample.name}, level={sample.level}): {len(educ_data)} variables')
            for k, ds in sorted(educ_data.items()):
                yrs = ds.years[:6]
                vals = [f'{ds.values[pd.Timestamp(y,1,1)]:.0f}' for y in yrs]
                print(f'  {list(ds.categories.values())[0]:50s}: years={yrs}, values={vals}')

M_educ_2000: 39310 DataSeries entries stored

Sample (Bolesławiec, level=6): 5 variables
  wyższe                                            : years=[2002, 2021], values=['3543', '7244']
  policealne oraz średnie zawodowe/branżowe         : years=[2002, 2021], values=['10147', '7017']
  średnie ogólnokształcące                          : years=[2002, 2021], values=['3325', '3940']
  zasadnicze zawodowe/branżowe                      : years=[2002, 2021], values=['8595', '7310']
  gimnazjalne, podstawowe i niższe                  : years=[2002, 2021], values=['10542', '6139']

Sample (bolesławiecki, level=5): 5 variables
  wyższe                                            : years=[2011], values=['8686']
  policealne oraz średnie zawodowe/branżowe         : years=[2011], values=['13524']
  średnie ogólnokształcące                          : years=[2011], values=['7596']
  zasadnicze zawodowe/branżowe                      : years=[2011], values=['20754']
  gimnazjalne, podstawowe i niższe 

In [22]:
# ── Cell 13: Create M_educ_sex_1990 ──
# P2402 (2002, gmina, all sex groups) + H_sex_educ (country, all sex groups)
# 2D: n1=educ (M_educ_1990 labels), n2=sex (ogółem, mężczyźni, kobiety)

SID_EDUC_SEX_1990 = 'M_educ_sex_1990'
SEX_LABELS = ['ogółem', 'mężczyźni', 'kobiety']

# P2402 → M_educ_1990 educ labels (policealne + średnie razem → średnie, etc.)
P2402_SEX1990_EDUC_MAP = {
    'wyższe': 'wyższe',
    'policealne': 'średnie',
    'średnie razem': 'średnie',
    'zasadnicze zawodowe': 'zasadnicze zawodowe',
    'podstawowe ukończone': 'podstawowe',
    'podstawowe nieukończone i bez wykształcenia': 'podstawowe nieukończone i bez wykształcenia',
}

# H_sex_educ → M_educ_1990 educ labels
H_EDUC_SEX1990_MAP = {
    'ogółem': 'ogółem',
    'wyższe': 'wyższe',
    'średnie': 'średnie',
    'zasadnicze zawodowe': 'zasadnicze zawodowe',
    'podstawowe': 'podstawowe',
    'niepełne podstawowe i bez wykształcenia': 'podstawowe nieukończone i bez wykształcenia',
}

n_created = 0
for tid, record in records.items():
    # ── P2402 (2002, level=6, all sex groups) ──
    if record.level == LEVEL_GMINA:
        pairs = extract_2d_all_sex(record, 'P2402', 'educ', P2402_SEX1990_EDUC_MAP)
        if pairs:
            # Compute ogółem educ for each sex group (sum of known categories)
            sex_seen = set(sl for (_, sl) in pairs.keys())
            for sex_lbl in sex_seen:
                og = pd.Series(data=np.nan, index=DATETIME_INDEX_FULL, dtype=float)
                for elbl in M_EDUC_1990_LABELS:
                    if elbl == 'ogółem':
                        continue
                    v = pairs.get((elbl, sex_lbl))
                    if v is not None:
                        _add_series_inplace(og, v)
                if og.notna().any():
                    pairs[('ogółem', sex_lbl)] = og
            
            n_created += store_2d_merged(record, SID_EDUC_SEX_1990,
                                          M_EDUC_1990_LABELS, SEX_LABELS, pairs, 'educ_sex_1990')
    
    # ── H_sex_educ (country level, all sex groups) ──
    h_data = record.get_data_by_subject('H_sex_educ')
    if h_data:
        pairs = extract_2d_all_sex(record, 'H_sex_educ', 'educ', H_EDUC_SEX1990_MAP)
        if pairs:
            # Compute 'niepełne' as residual for years/sex combos where not available
            sex_seen = set(sl for (_, sl) in pairs.keys())
            for sex_lbl in sex_seen:
                og = pairs.get(('ogółem', sex_lbl))
                pn_key = ('podstawowe nieukończone i bez wykształcenia', sex_lbl)
                if og is not None and pn_key not in pairs:
                    residual = pd.Series(data=np.nan, index=DATETIME_INDEX_FULL, dtype=float)
                    for ts in og.index:
                        og_val = og.get(ts, np.nan)
                        if pd.isna(og_val):
                            continue
                        parts = 0.0
                        for elbl in ['wyższe', 'średnie', 'zasadnicze zawodowe', 'podstawowe']:
                            v = pairs.get((elbl, sex_lbl), pd.Series(dtype=float)).get(ts, 0)
                            if pd.isna(v): v = 0
                            parts += v
                        res = og_val - parts
                        if res >= 0:
                            residual[ts] = res
                    if residual.notna().any():
                        pairs[pn_key] = residual
            
            n_created += store_2d_merged(record, SID_EDUC_SEX_1990,
                                          M_EDUC_1990_LABELS, SEX_LABELS, pairs, 'educ_sex_1990')

print(f'M_educ_sex_1990: {n_created} DataSeries entries stored')

# Validation
sample = records['0201011']
data = sample.get_data_by_subject(SID_EDUC_SEX_1990)
print(f'\nSample ({sample.name}): {len(data)} series')
for k, ds in sorted(data.items())[:6]:
    yrs = ds.years
    print(f'  {ds.categories}: years={yrs}')

M_educ_sex_1990: 65664 DataSeries entries stored

Sample (Bolesławiec): 18 series
  {'n1': 'ogółem', 'n2': 'ogółem'}: years=[2002]
  {'n1': 'ogółem', 'n2': 'mężczyźni'}: years=[2002]
  {'n1': 'ogółem', 'n2': 'kobiety'}: years=[2002]
  {'n1': 'wyższe', 'n2': 'ogółem'}: years=[2002]
  {'n1': 'wyższe', 'n2': 'mężczyźni'}: years=[2002]
  {'n1': 'wyższe', 'n2': 'kobiety'}: years=[2002]


In [23]:
# ── Cell 14: Create M_educ_sex_2000 ──
# P2402 (2002, gmina) + P3309 (2011, powiat) + P4315 (2021, gmina) — all sex groups
# 2D: n1=educ (M_educ_2000 labels), n2=sex

SID_EDUC_SEX_2000 = 'M_educ_sex_2000'

# Same mapping as M_educ_2000, but now keeping all sex groups
P2402_SEX2000_MAP = P2402_2000_MAP  # reuse from Cell 12
P3309_SEX2000_MAP = P3309_2000_MAP
P4315_SEX2000_MAP = P4315_2000_MAP

n_created = 0
for tid, record in records.items():
    # P2402 (2002, gmina, all sex) — priority
    if record.level == LEVEL_GMINA:
        pairs = extract_2d_all_sex(record, 'P2402', 'educ', P2402_SEX2000_MAP)
        if pairs:
            n_created += store_2d_merged(record, SID_EDUC_SEX_2000,
                                          M_EDUC_2000_LABELS, SEX_LABELS, pairs, 'educ_sex_2000')
    
    # P3309 (2011, powiat, all sex)
    if record.level == LEVEL_POWIAT:
        pairs = extract_2d_all_sex(record, 'P3309', 'educ', P3309_SEX2000_MAP)
        if pairs:
            n_created += store_2d_merged(record, SID_EDUC_SEX_2000,
                                          M_EDUC_2000_LABELS, SEX_LABELS, pairs, 'educ_sex_2000')
    
    # P4315 (2021, gmina, all sex)
    if record.level == LEVEL_GMINA:
        pairs = extract_2d_all_sex(record, 'P4315', 'educ', P4315_SEX2000_MAP)
        if pairs:
            n_created += store_2d_merged(record, SID_EDUC_SEX_2000,
                                          M_EDUC_2000_LABELS, SEX_LABELS, pairs, 'educ_sex_2000')

print(f'M_educ_sex_2000: {n_created} DataSeries entries stored')

# Validation
for sample_tid in ['0201011', '0201000']:
    sample = records.get(sample_tid)
    if sample:
        data = sample.get_data_by_subject(SID_EDUC_SEX_2000)
        if data:
            print(f'\nSample ({sample.name}, level={sample.level}): {len(data)} series')
            for k, ds in sorted(data.items())[:6]:
                yrs = ds.years
                print(f'  {ds.categories}: years={yrs}')

M_educ_sex_2000: 117360 DataSeries entries stored

Sample (Bolesławiec, level=6): 15 series
  {'n1': 'wyższe', 'n2': 'ogółem'}: years=[2002, 2021]
  {'n1': 'wyższe', 'n2': 'mężczyźni'}: years=[2002, 2021]
  {'n1': 'wyższe', 'n2': 'kobiety'}: years=[2002, 2021]
  {'n1': 'policealne oraz średnie zawodowe/branżowe', 'n2': 'ogółem'}: years=[2002, 2021]
  {'n1': 'policealne oraz średnie zawodowe/branżowe', 'n2': 'mężczyźni'}: years=[2002, 2021]
  {'n1': 'policealne oraz średnie zawodowe/branżowe', 'n2': 'kobiety'}: years=[2002, 2021]

Sample (bolesławiecki, level=5): 15 series
  {'n1': 'wyższe', 'n2': 'ogółem'}: years=[2011]
  {'n1': 'wyższe', 'n2': 'mężczyźni'}: years=[2011]
  {'n1': 'wyższe', 'n2': 'kobiety'}: years=[2011]
  {'n1': 'policealne oraz średnie zawodowe/branżowe', 'n2': 'ogółem'}: years=[2011]
  {'n1': 'policealne oraz średnie zawodowe/branżowe', 'n2': 'mężczyźni'}: years=[2011]
  {'n1': 'policealne oraz średnie zawodowe/branżowe', 'n2': 'kobiety'}: years=[2011]


## Step 4: Build cross tables for all new M_ subjects

In [24]:
# ── Cell 15: Build cross tables for new M_ subjects ──
new_subjects = [
    (SID_HH1990, 'hh_size_1990'),
    (SID_HH2000, 'hh_size_2000'),
    (SID_AGE1990, 'age_1990'),
    (SID_EDUC1990, 'educ_1990'),
    (SID_EDUC2000, 'educ_2000'),
    (SID_EDUC_SEX_1990, 'educ_sex_1990'),
    (SID_EDUC_SEX_2000, 'educ_sex_2000'),
]

for sid, name in new_subjects:
    print(f'\n── Building cross tables for {sid} ──')
    count = db.build_cross_tables(sid, subject_name=name, verbose=True)
    print(f'   Built {count} cross tables')


── Building cross tables for M_hh_size_1990 ──
  ✓ Built 3724 cross tables for subject M_hh_size_1990 (hh_size_1990)
   Built 3724 cross tables

── Building cross tables for M_hh_size_2000 ──
  ✓ Built 4257 cross tables for subject M_hh_size_2000 (hh_size_2000)
   Built 4257 cross tables

── Building cross tables for M_age_1990 ──
  ✓ Built 4184 cross tables for subject M_age_1990 (age_1990)
   Built 4184 cross tables

── Building cross tables for M_educ_1990 ──
  ✓ Built 3725 cross tables for subject M_educ_1990 (educ_1990)
   Built 3725 cross tables

── Building cross tables for M_educ_2000 ──
  ✓ Built 4276 cross tables for subject M_educ_2000 (educ_2000)
   Built 4276 cross tables

── Building cross tables for M_educ_sex_1990 ──
  ✓ Built 3648 cross tables for subject M_educ_sex_1990 (educ_sex_1990)
   Built 3648 cross tables

── Building cross tables for M_educ_sex_2000 ──
  ✓ Built 4257 cross tables for subject M_educ_sex_2000 (educ_sex_2000)
   Built 4257 cross tables


## Step 5: Comprehensive validation

Verify:
1. Year range extension worked correctly
2. All M_ subjects have the expected labels and data coverage
3. Cross tables are correctly built with proper dimensions
4. Data coverage matrix per year

In [25]:
# ── Cell 16: Comprehensive validation ──
print('='*80)
print('VALIDATION REPORT — v5.0 Prerequisites')
print('='*80)

# 1. Year range
print('\n1. YEAR RANGE CHECK')
sample = records['0201011']
print(f'   Constants: {YEAR_RANGE_FULL[0]}–{YEAR_RANGE_FULL[-1]} ({len(YEAR_RANGE_FULL)} years)')
print(f'   Sample pop index: {sample.pop.index[0].year}–{sample.pop.index[-1].year} ({len(sample.pop)} entries)')
sample_key = list(sample.data.keys())[0]
print(f'   Sample DataSeries: {sample.data[sample_key].values.index[0].year}–{sample.data[sample_key].values.index[-1].year}')

# 2. New M_ subjects overview
print('\n2. NEW M_ SUBJECTS OVERVIEW')
new_sids = [SID_HH1990, SID_HH2000, SID_AGE1990, SID_EDUC1990, SID_EDUC2000, 
            SID_EDUC_SEX_1990, SID_EDUC_SEX_2000]

for sid in new_sids:
    n_records_with_ct = 0
    n_records_with_ds = 0
    sample_ct = None
    levels = defaultdict(int)
    
    for r in records.values():
        if r.get_data_by_subject(sid):
            n_records_with_ds += 1
        if sid in r.cross_tables:
            ct = r.cross_tables[sid]
            if ct.years_with_data:
                n_records_with_ct += 1
                levels[r.level] += 1
                if sample_ct is None:
                    sample_ct = ct
    
    lvl_str = ', '.join(f'L{l}={c}' for l, c in sorted(levels.items()))
    if sample_ct:
        print(f'\n   {sid}:')
        print(f'     Records with DataSeries: {n_records_with_ds}')
        print(f'     Records with CrossTable: {n_records_with_ct} [{lvl_str}]')
        print(f'     Dimensions: {sample_ct.dim_names}')
        print(f'     Shape: {sample_ct.shape}')
        for i, d in enumerate(sample_ct.dim_names):
            print(f'       {d}: {sample_ct.dim_labels[d]}')
        print(f'     Years with data: {sample_ct.years_with_data[:10]}...')
    else:
        print(f'\n   {sid}: NO cross tables built (DataSeries in {n_records_with_ds} records)')

# 3. Data coverage matrix (gminas)
print('\n\n3. DATA COVERAGE PER YEAR (gminas, level=6)')
total_gminas = sum(1 for r in records.values() if r.level == LEVEL_GMINA)
print(f'   Total gminas: {total_gminas}')

for sid in new_sids:
    year_counts = defaultdict(int)
    for r in records.values():
        if r.level != LEVEL_GMINA:
            continue
        if sid in r.cross_tables:
            for yr in r.cross_tables[sid].years_with_data:
                year_counts[yr] += 1
    if year_counts:
        print(f'\n   {sid}:')
        for yr in sorted(year_counts.keys()):
            pct = year_counts[yr] / total_gminas * 100
            bar = '█' * int(pct / 2)
            print(f'      {yr}: {year_counts[yr]:5d}/{total_gminas} ({pct:5.1f}%) {bar}')

# 4. Consistency check: sum of sub-categories == ogółem (for 1D subjects with ogółem)
print('\n\n4. CONSISTENCY: sub-category sum vs ogółem')
for sid in [SID_HH1990, SID_AGE1990, SID_EDUC1990]:
    n_checked = 0
    n_match = 0
    n_mismatch = 0
    max_error = 0.0
    
    for r in records.values():
        if r.level != LEVEL_GMINA:
            continue
        if sid not in r.cross_tables:
            continue
        ct = r.cross_tables[sid]
        if 'ogółem' not in ct.dim_labels.get('n1', []):
            continue
        
        og_idx = ct.dim_labels['n1'].index('ogółem')
        
        for yr in ct.years_with_data:
            tbl = ct.tables[yr]
            if ct.ndim == 1:
                og_val = tbl[og_idx]
                sub_sum = np.nansum([tbl[i] for i in range(len(tbl)) if i != og_idx])
            else:
                og_val = tbl[og_idx]  # for 1D, tbl is 1D array
                sub_sum = np.nansum([tbl[i] for i in range(len(tbl)) if i != og_idx])
            
            if np.isnan(og_val) or np.isnan(sub_sum):
                continue
            n_checked += 1
            error = abs(og_val - sub_sum)
            max_error = max(max_error, error)
            if error < 1.0:
                n_match += 1
            else:
                n_mismatch += 1
    
    print(f'   {sid}: checked {n_checked}, match={n_match}, mismatch={n_mismatch}, max_error={max_error:.1f}')

print('\n' + '='*80)
print('VALIDATION COMPLETE')
print('='*80)

VALIDATION REPORT — v5.0 Prerequisites

1. YEAR RANGE CHECK
   Constants: 1986–2025 (40 years)
   Sample pop index: 1986–2025 (40 entries)
   Sample DataSeries: 1986–2025

2. NEW M_ SUBJECTS OVERVIEW

   M_hh_size_1990:
     Records with DataSeries: 3724
     Records with CrossTable: 3724 [L6=3724]
     Dimensions: ['n1']
     Shape: (5,)
       n1: ['1-osobowe', '2-osobowe', '3-4-osobowe', '5 i więcej-osobowe', 'ogółem']
     Years with data: [1988, 2002]...

   M_hh_size_2000:
     Records with DataSeries: 4257
     Records with CrossTable: 4257 [L5=379, L6=3878]
     Dimensions: ['n1']
     Shape: (6,)
       n1: ['1-osobowe', '2-osobowe', '3-osobowe', '4-osobowe', '5 i więcej-osobowe', 'ogółem']
     Years with data: [2011]...

   M_age_1990:
     Records with DataSeries: 4184
     Records with CrossTable: 4184 [L0=1, L2=49, L6=4134]
     Dimensions: ['n1']
     Shape: (8,)
       n1: ['0-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60 lat i więcej', 'ogółem']
     Years with 

## Save updated database

In [26]:
# ── Cell 17: Save updated database ──
SAVE_PATH = DB_PATH.parent / 'geoteryt_v5.pkl'
print(f'Saving to: {SAVE_PATH}')
db.save_complete(SAVE_PATH, verbose=True)
print(f'\nDone! Database saved with v5.0 prerequisites.')

Saving to: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_v5.pkl
Saving complete database to /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_v5.pkl...
  ✓ Saved 4612 records
  ✓ Records with data: 4584
  ✓ Records with cross tables: 4584
  ✓ File size: 1706.8 MB
  ✓ Path: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_v5.pkl

Done! Database saved with v5.0 prerequisites.
